# Human Face Recognition using Deep Learning

celebrity face recognition system using FaceNet embeddings, MTCNN face detection, and a TensorFlow/Keras classifier. It is written to run in Google Colab from top to bottom.

In [ ]:
# Cell 1: Upload kaggle.json,  download celebrity dataset, unzip, configure Kaggle, and organize folders
import os
import shutil
import zipfile
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path('/content/Human-Face-Recognition') if IN_COLAB else Path.cwd() / 'Human-Face-Recognition'
DATASET_DIR = BASE_DIR / 'dataset'
RAW_DIR = DATASET_DIR / 'raw'
STRUCTURED_DIR = DATASET_DIR / 'structured'
MODELS_DIR = BASE_DIR / 'models'

for folder in [BASE_DIR, DATASET_DIR, RAW_DIR, STRUCTURED_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = 'bhaveshmittal/celebrity-face-recognition-dataset'
ZIP_PATH = DATASET_DIR / 'celebrity-face-recognition-dataset.zip'

if not IN_COLAB:
    raise EnvironmentError('This notebook is intended for Google Colab. Open it in Colab and run again.')

from google.colab import files
print('Upload kaggle.json to continue...')
uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('kaggle.json not uploaded. Rerun this cell and upload the file.')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
if not ZIP_PATH.exists():
    subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', str(DATASET_DIR)], check=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(RAW_DIR)

image_exts = {'.jpg', '.jpeg', '.png', '.webp'}
for img_path in RAW_DIR.rglob('*'):
    if img_path.is_file() and img_path.suffix.lower() in image_exts:
        class_name = img_path.parent.name.strip().replace('  ', ' ')
        target_dir = STRUCTURED_DIR / class_name
        target_dir.mkdir(parents=True, exist_ok=True)
        target_path = target_dir / img_path.name
        if not target_path.exists():
            shutil.copy2(img_path, target_path)

class_dirs = [p for p in STRUCTURED_DIR.iterdir() if p.is_dir()]
print('Structured dataset ready at:', STRUCTURED_DIR)
print('Total celebrity classes found:', len(class_dirs))
print('Sample classes:', [p.name for p in class_dirs[:10]])

Upload kaggle.json to continue...


Saving kaggle.json to kaggle.json


In [ ]:
# Cell 2: Install dependencies
import sys
import subprocess

packages = [
    'numpy', 'pandas', 'matplotlib', 'seaborn', 'opencv-python', 'tensorflow',
    'scikit-learn', 'joblib', 'tqdm', 'mtcnn', 'keras-facenet', 'deepface', 'requests'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

Dependencies installed.


In [ ]:
# Cell 3: Imports and configuration
import io
import json
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from keras import Sequential
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.layers import BatchNormalization, Dense, Dropout, Input
from keras.models import load_model
from keras_facenet import FaceNet
from mtcnn import MTCNN
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(gpus))
if gpus:
    print(gpus)

@dataclass
class Config:
    image_size: tuple = (160, 160)
    min_images_per_class: int = 20
    max_classes: int = 17
    test_size: float = 0.15
    val_size: float = 0.15
    batch_size: int = 32
    epochs: int = 10
    unknown_distance_threshold: float = 0.40
    classifier_confidence_threshold: float = 0.60

CFG = Config()
detector = MTCNN()
embedder = FaceNet()

TensorFlow version: 2.20.0
GPU available: False


In [ ]:
# Cell 5: EDA(Exploratory Data Analysis ) This cell defines helper functions for preparing and inspecting the dataset.
plot_distribution(df)
show_samples(df)

NameError: name 'plot_distribution' is not defined

In [ ]:
# Cell 6: Train, validation, test split
train_df, test_df = train_test_split(df, test_size=CFG.test_size, random_state=SEED, stratify=df['label'])
val_ratio_adjusted = CFG.val_size / (1 - CFG.test_size)
train_df, val_df = train_test_split(train_df, test_size=val_ratio_adjusted, random_state=SEED, stratify=train_df['label'])
print('Train:', train_df.shape)
print('Validation:', val_df.shape)
print('Test:', test_df.shape)

In [ ]:
# Cell 7: Face detection, alignment, and embedding functions in this cell each actor face align with their sizes like distance between two eye , ears and other main point s
def align_face(image_bgr, box, keypoints, margin=12):
    x, y, w, h = box
    x = max(0, x - margin)
    y = max(0, y - margin)
    x2 = min(image_bgr.shape[1], x + w + 2 * margin)
    y2 = min(image_bgr.shape[0], y + h + 2 * margin)
    left_eye = keypoints.get('left_eye')
    right_eye = keypoints.get('right_eye')
    if left_eye and right_eye:
        eye_center = ((left_eye[0] + right_eye[0]) / 2, (left_eye[1] + right_eye[1]) / 2)
        angle = np.degrees(np.arctan2(right_eye[1] - left_eye[1], right_eye[0] - left_eye[0]))
        rotation_matrix = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
        image_bgr = cv2.warpAffine(image_bgr, rotation_matrix, (image_bgr.shape[1], image_bgr.shape[0]), flags=cv2.INTER_CUBIC)
    face = image_bgr[y:y2, x:x2]
    if face.size == 0:
        return None
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face = cv2.resize(face, CFG.image_size)
    return face

def extract_single_face(image_path):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        return None, 'image_read_error'
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    try:
        detections = detector.detect_faces(image_rgb)
    except ValueError as e:
        # Catch the ValueError specifically from the MTCNN Conv2D if it occurs
        return None, f'mtcnn_detection_error: {e}'
    if not detections:
        return None, 'no_face_detected'
    best = max(detections, key=lambda d: d['box'][2] * d['box'][3])
    aligned = align_face(image_bgr, best['box'], best['keypoints'])
    if aligned is None:
        return None, 'alignment_failed'
    return aligned, None

def build_embeddings(split_df, split_name):
    embeddings, labels, faces, failures = [], [], [], []
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        face, error = extract_single_face(row['image_path'])
        if face is None:
            failures.append({'image_path': row['image_path'], 'label': row['label'], 'error': error})
            continue
        try:
            embedding = embedder.embeddings([face])[0]
            embeddings.append(embedding)
            labels.append(row['label'])
            faces.append(face)
        except Exception as exc:
            failures.append({'image_path': row['image_path'], 'label': row['label'], 'error': str(exc)})
    return {
        'X': np.array(embeddings),
        'y': np.array(labels),
        'faces': np.array(faces),
        'failures': pd.DataFrame(failures)
    }

In [ ]:
train_data = build_embeddings(train_df, 'train')
val_data = build_embeddings(val_df, 'validation')
test_data = build_embeddings(test_df, 'test')

print('Train embeddings:', train_data['X'].shape)
print('Validation embeddings:', val_data['X'].shape)
print('Test embeddings:', test_data['X'].shape)
print('Total failures:', len(train_data['failures']) + len(val_data['failures']) + len(test_data['failures']))

if not train_data['failures'].empty:
    print('\nTrain failures:')
    display(train_data['failures'].head())
if not val_data['failures'].empty:
    print('\nValidation failures:')
    display(val_data['failures'].head())
if not test_data['failures'].empty:
    print('\nTest failures:')
    display(test_data['failures'].head())

In [ ]:
# Cell 9: Visualize aligned faces  This function takes a detected face and aligns it. why it is important this cell If one image is sideways and another is straight, embeddings become less consistent

sample_faces = train_data['faces'][:12]
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, face in zip(axes.flat, sample_faces):
    ax.imshow(face)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Encode labels, scale embeddings, define and train classifier  This cell actually runs the face processing pipeline on all images.
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_data['y'])
y_val = label_encoder.transform(val_data['y'])
y_test = label_encoder.transform(test_data['y'])

scaler = StandardScaler()
X_train = scaler.fit_transform(train_data['X'])
X_val = scaler.transform(val_data['X'])
X_test = scaler.transform(test_data['X'])

num_classes = len(label_encoder.classes_)
classifier = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.35),
    Dense(128, activation='relu'),
    Dropout(0.25),
    Dense(num_classes, activation='softmax')
])
classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    ModelCheckpoint(MODELS_DIR / 'best_face_classifier.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
]

history = classifier.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CFG.epochs,
    batch_size=CFG.batch_size,
    callbacks=callbacks,
    verbose=1
)

joblib.dump(label_encoder, MODELS_DIR / 'label_encoder.pkl')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
print('Model and preprocessing artifacts saved.')

In [ ]:
# Cell 11: Training graphs
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df['accuracy'], label='train')
axes[0].plot(history_df['val_accuracy'], label='val')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[1].plot(history_df['loss'], label='train')
axes[1].plot(history_df['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Evaluation and confusion matrix
best_model = load_model(MODELS_DIR / 'best_face_classifier.keras')
y_prob = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print('\nClassification report:\n')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

class_centroids = []
for class_name in label_encoder.classes_:
    class_embeddings = train_data['X'][train_data['y'] == class_name]
    class_centroids.append(class_embeddings.mean(axis=0))
class_centroids = np.array(class_centroids)
np.save(MODELS_DIR / 'class_centroids.npy', class_centroids)
np.save(MODELS_DIR / 'class_names.npy', label_encoder.classes_)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Cell 13: PCA(Principal Component Analysis) visualization of FaceNet embeddings
sample_n = min(300, len(train_data['X']))
idx = np.random.choice(len(train_data['X']), sample_n, replace=False)
sample_X = train_data['X'][idx]
sample_y = train_data['y'][idx]

pca = PCA(n_components=2, random_state=SEED)
pca_2d = pca.fit_transform(sample_X)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
sns.scatterplot(x=pca_2d[:, 0], y=pca_2d[:, 1], hue=sample_y, ax=ax, palette='tab10')
ax.set_title('PCA of embeddings')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Inference helpers for uploaded image, image URL, unknown detection, and attendance
ATTENDANCE_CSV = MODELS_DIR / 'attendance.csv'

def mark_attendance(name):
    if name == 'Unknown':
        return
    timestamp = pd.Timestamp.now()
    row = {'name': name, 'date': timestamp.date().isoformat(), 'time': timestamp.strftime('%H:%M:%S')}
    if ATTENDANCE_CSV.exists():
        df_att = pd.read_csv(ATTENDANCE_CSV)
        already = ((df_att['name'] == row['name']) & (df_att['date'] == row['date'])).any()
        if already:
            return
        df_att = pd.concat([df_att, pd.DataFrame([row])], ignore_index=True)
    else:
        df_att = pd.DataFrame([row])
    df_att.to_csv(ATTENDANCE_CSV, index=False)

def predict_faces_from_rgb(image_rgb):
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    detections = detector.detect_faces(image_rgb)
    results = []
    annotated = image_rgb.copy()
    for det in detections:
        aligned = align_face(image_bgr, det['box'], det['keypoints'])
        if aligned is None:
            continue
        embedding = embedder.embeddings([aligned])[0]
        scaled = scaler.transform([embedding])
        probs = best_model.predict(scaled, verbose=0)[0]
        pred_idx = int(np.argmax(probs))
        pred_name = label_encoder.inverse_transform([pred_idx])[0]
        pred_conf = float(probs[pred_idx])
        centroid_idx = int(np.where(label_encoder.classes_ == pred_name)[0][0])
        distance = float(cosine_distances([embedding], [class_centroids[centroid_idx]])[0][0])
        if distance > CFG.unknown_distance_threshold or pred_conf < CFG.classifier_confidence_threshold:
            final_name = 'Unknown'
            final_conf = max(0.0, 1.0 - distance)
        else:
            final_name = pred_name
            final_conf = pred_conf
            mark_attendance(final_name)
        x, y, w, h = det['box']
        color = (0, 255, 0) if final_name != 'Unknown' else (255, 60, 60)
        cv2.rectangle(annotated, (x, y), (x + w, y + h), color, 2)
        cv2.putText(annotated, f'{final_name} | {final_conf:.2f}', (x, max(25, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        results.append({'identity': final_name, 'confidence': final_conf, 'distance': distance, 'box': det['box']})
    return annotated, pd.DataFrame(results)

def load_image_from_url(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    arr = np.frombuffer(response.content, dtype=np.uint8)
    image_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise ValueError('Could not decode image from URL.')
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# Cell 15: Test on uploaded custom image
from google.colab import files
uploaded = files.upload()
if uploaded:
    file_name = next(iter(uploaded))
    file_bytes = np.frombuffer(uploaded[file_name], np.uint8)
    image_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    annotated, result_df = predict_faces_from_rgb(image_rgb)
    plt.figure(figsize=(10, 8))
    plt.imshow(annotated)
    plt.axis('off')
    plt.show()
    display(result_df)